# ANON4REID - L2 SD Inpainting Dataset Generation (Kaggle 2x T4)

This notebook generates the **Level 2 (SD Inpainting)** anonymized dataset for Market-1501 using **both available T4 GPUs** in parallel to halve processing time.

**Pipeline:** Each person image is upscaled from 128x64 to 512x256, a geometric mask covering the central torso region (60% width x 80% height) is applied, and Stable Diffusion Inpainting replaces the masked person area with diffusion-generated content conditioned on the prompt "empty background, no person, floor, wall". The result is then downscaled back to 128x64.

**Dual-GPU strategy:** Two pipeline instances are loaded -- one on `cuda:0`, one on `cuda:1`. Each split's images are divided into two chunks and processed concurrently via Python threads (CUDA releases the GIL, so threads run truly in parallel on separate GPUs). Skip-if-exists resumability is preserved.

**Splits processed:** `query` (3,368 images), `bounding_box_test` (19,732 images), `bounding_box_train` (12,936 images) -- total ~36,036 images.

**Output:** `/kaggle/working/Market-1501-SDInpaint/` -- directory structure mirrors Market-1501 so it can be used as a drop-in replacement for evaluation. A zip archive is created for easy download.


In [ ]:
!pip install diffusers transformers accelerate peft

In [ ]:
import os
import shutil
import threading
import torch
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from diffusers import StableDiffusionInpaintPipeline

print("Libraries loaded successfully.")
print(f"PyTorch version: {torch.__version__}")
gpu_count = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {gpu_count}")
for i in range(gpu_count):
    print(f"  cuda:{i}: {torch.cuda.get_device_name(i)}")


## Step 0 — Discover the dataset mount path

Run this cell to see where Kaggle mounted the Market-1501 dataset and **auto-detect** the correct `INPUT_DIR`. The split directories (`query/`, `bounding_box_test/`, `bounding_box_train/`) may be nested under a subfolder like `Market-1501-v15.09.15/` rather than directly at the mount root. This cell sets `INPUT_DIR` as a global variable that the processing cell below will use.

In [ ]:
# Discover the dataset structure under the Kaggle mount path
MOUNT_BASE = "/kaggle/input/datasets/rayiooo/reid_market-1501"

print(f"Contents of {MOUNT_BASE}:")
for entry in sorted(os.listdir(MOUNT_BASE)):
    full = os.path.join(MOUNT_BASE, entry)
    if os.path.isdir(full):
        subdirs = [d for d in os.listdir(full) if os.path.isdir(os.path.join(full, d))]
        print(f"  {entry}/  ->  subdirs: {subdirs}")
    else:
        print(f"  {entry}")

# Auto-detect the Market-1501 root (the directory containing query/, bounding_box_test/, bounding_box_train/)
INPUT_DIR = MOUNT_BASE
for entry in os.listdir(MOUNT_BASE):
    candidate = os.path.join(MOUNT_BASE, entry)
    if os.path.isdir(candidate):
        subdirs = os.listdir(candidate)
        if "query" in subdirs and "bounding_box_train" in subdirs:
            INPUT_DIR = candidate
            print(f"\nAuto-detected INPUT_DIR = {INPUT_DIR}")
            break

# Verify splits exist
for split in ["query", "bounding_box_test", "bounding_box_train"]:
    split_path = os.path.join(INPUT_DIR, split)
    exists = os.path.exists(split_path)
    count = len([f for f in os.listdir(split_path) if f.endswith(".jpg")]) if exists else 0
    print(f"  {split}: exists={exists}, .jpg count={count}")


In [ ]:
# Scaling constants (must match src/anonymizer.py exactly)
ORIGINAL_SIZE = (64, 128)   # (width, height) -- native Market-1501 resolution
MODEL_SIZE    = (256, 512)  # (width, height) -- SD Inpainting working resolution


def upscale(image: Image.Image) -> Image.Image:
    """Upscale image to MODEL_SIZE using LANCZOS interpolation."""
    return image.resize(MODEL_SIZE, Image.LANCZOS)


def downscale(image: Image.Image) -> Image.Image:
    """Downscale image to ORIGINAL_SIZE using LANCZOS interpolation."""
    return image.resize(ORIGINAL_SIZE, Image.LANCZOS)


def preprocess(image: Image.Image) -> Image.Image:
    """Convert to RGB and upscale to model working resolution."""
    image = image.convert("RGB")
    image = upscale(image)
    return image


def postprocess(image: Image.Image) -> Image.Image:
    """Downscale back to original Market-1501 resolution."""
    return downscale(image)


# Mask generation (must match src/sd_inpaint.py exactly)

def generate_person_mask(image: Image.Image) -> Image.Image:
    """
    Generates a binary mask covering the central person region of the image.
    Since Market-1501 images are tightly cropped around the person, masking
    the central 60% width and 80% height reliably covers the subject without
    needing a separate segmentation model. The mask is white (255) where the
    person is and black (0) everywhere else, which is the format the inpainting
    model expects.
    """
    w, h = image.size
    mask = np.zeros((h, w), dtype=np.uint8)
    x1, x2 = int(w * 0.2), int(w * 0.8)
    y1, y2 = int(h * 0.1), int(h * 0.9)
    mask[y1:y2, x1:x2] = 255
    return Image.fromarray(mask).convert("RGB")


# Pipeline loader (must match src/sd_inpaint.py exactly)

def load_sd_inpaint_pipeline(device: str = "cuda"):
    """Load StableDiffusionInpaintPipeline with fp16 on CUDA, no safety checker."""
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "sd-legacy/stable-diffusion-inpainting",
        torch_dtype=torch.float16 if "cuda" in device else torch.float32,
        safety_checker=None,
    ).to(device)
    pipe.enable_attention_slicing()
    print(f"SD Inpaint pipeline loaded on {device}.")
    return pipe


# Anonymization function (must match src/sd_inpaint.py exactly)

def anonymize_sd_inpaint(
    image: Image.Image,
    pipe,
    prompt: str = "empty background, no person, floor, wall",
    negative_prompt: str = "person, human, body, blurry, distorted",
    device: str = "cuda"
) -> Image.Image:
    """
    Takes a single PIL Image, runs the full Stable Diffusion Inpainting
    anonymization pipeline, and returns the anonymized image at the original
    128x64 resolution. Handles upscaling, mask generation, inpainting, and
    downscaling internally.
    """
    image_up = preprocess(image)
    mask = generate_person_mask(image_up)

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image_up,
        mask_image=mask,
        height=512,
        width=256,
        num_inference_steps=20,
        guidance_scale=7.5,
    ).images[0]

    return postprocess(result)


print("All pipeline functions defined.")

In [ ]:
# Path configuration
# INPUT_DIR was auto-detected by the discovery cell above.
# If auto-detection failed (no print output), manually set it here:
#   INPUT_DIR = "/kaggle/input/datasets/rayiooo/reid_market-1501/Market-1501-v15.09.15"
OUTPUT_DIR = "/kaggle/working/Market-1501-SDInpaint"
SPLITS     = ["query", "bounding_box_test", "bounding_box_train"]

# Detect available GPUs
GPU_COUNT = torch.cuda.device_count()
DEVICES   = [f"cuda:{i}" for i in range(GPU_COUNT)] if GPU_COUNT > 0 else ["cpu"]
print(f"GPUs detected: {GPU_COUNT}")
print(f"Devices: {DEVICES}")

# Create output directories
for split in SPLITS:
    os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

print(f"Input : {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Splits: {SPLITS}")

# Load one pipeline per GPU
print("\nLoading SD Inpaint pipelines...")
pipes = []
for dev in DEVICES:
    pipes.append(load_sd_inpaint_pipeline(dev))


# ── Dual-GPU processing ──────────────────────────────────────────────────

def get_images(split_dir: str):
    """Return list of .jpg filenames in a split directory."""
    return [f for f in os.listdir(split_dir) if f.endswith(".jpg")]


def worker_fn(images, pipe, device, input_split_dir, output_split_dir, progress_bar):
    """Process a chunk of images on a specific GPU. Thread-safe via per-GPU pipeline."""
    local_processed = 0
    local_skipped   = 0
    for fname in images:
        input_path  = os.path.join(input_split_dir, fname)
        output_path = os.path.join(output_split_dir, fname)

        # Skip if already processed -- enables resumability after interruptions
        if os.path.exists(output_path):
            local_skipped += 1
            progress_bar.update(1)
            continue

        try:
            img    = Image.open(input_path).convert("RGB")
            result = anonymize_sd_inpaint(img, pipe, device=device)
            result.save(output_path)
            local_processed += 1
        except Exception as e:
            print(f"\nError processing {fname} on {device}: {e}, skipping...")

        progress_bar.update(1)

    return local_processed, local_skipped


def process_split(split: str, pipes, devices):
    """Process all images in a split across available GPUs in parallel."""
    input_split_dir  = os.path.join(INPUT_DIR, split)
    output_split_dir = os.path.join(OUTPUT_DIR, split)

    if not os.path.exists(input_split_dir):
        print(f"Skipping {split} -- directory not found at {input_split_dir}")
        return

    images = get_images(input_split_dir)
    n_gpus = len(devices)
    print(f"\nProcessing {split}: {len(images)} images across {n_gpus} GPU(s)")

    # Distribute images across GPUs
    chunks = [[] for _ in range(n_gpus)]
    for i, fname in enumerate(images):
        chunks[i % n_gpus].append(fname)

    # Shared progress bar for all threads
    pbar = tqdm(total=len(images), desc=split)

    # Launch threads -- one per GPU
    threads = []
    results = {}
    for gpu_idx, (chunk, pipe, device) in enumerate(zip(chunks, pipes, devices)):
        t = threading.Thread(
            target=lambda ci, ch, p, d, res_dict: res_dict.update(
                {ci: worker_fn(ch, p, d, input_split_dir, output_split_dir, pbar)}
            ),
            args=(gpu_idx, chunk, pipe, device, results)
        )
        threads.append(t)
        t.start()

    # Wait for all threads to finish
    for t in threads:
        t.join()

    pbar.close()

    total_processed = sum(r[0] for r in results.values())
    total_skipped   = sum(r[1] for r in results.values())
    print(f"  Done -- {total_processed} processed, {total_skipped} skipped (already existed)")


# Run all 3 splits sequentially (each split uses all GPUs in parallel)
total_images = sum(
    len(get_images(os.path.join(INPUT_DIR, s)))
    for s in SPLITS
    if os.path.exists(os.path.join(INPUT_DIR, s))
)
print(f"\nTotal images to process: {total_images}")

for split in SPLITS:
    process_split(split, pipes, DEVICES)

print(f"\n{'='*50}")
print(f"SD Inpaint dataset generation complete!")
print(f"Output saved to: {OUTPUT_DIR}")
print(f"{'='*50}")

In [ ]:
# Verification: per-split file counts
print("Verification -- per-split file counts:")
total = 0
for split in SPLITS:
    split_dir = os.path.join(OUTPUT_DIR, split)
    count = len([f for f in os.listdir(split_dir) if f.endswith(".jpg")])
    print(f"  {split}: {count} images")
    total += count
print(f"  Total: {total} images (expected ~36,036)")


In [ ]:
# Zip for download
print("Zipping Market-1501-SDInpaint...")
archive_path = shutil.make_archive("/kaggle/working/Market-1501-SDInpaint", "zip", OUTPUT_DIR)
print(f"Archive created: {archive_path}")
print("\nDownload instructions:")
print("1. Go to the Kaggle notebook Output panel")
print("2. Find 'Market-1501-SDInpaint.zip'")
print("3. Click the download icon")
print("4. Unzip to data/processed/Market-1501-SDInpaint/ in your local project")
